In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib
from numpy import random
import copy
import scipy
import os
import sys

In [ ]:
matplotlib.rc("font", **{"family":"serif","serif":["Computer Modern Roman"]})
matplotlib.rc("text", usetex=True)

# load data

In [ ]:
# simulation parameters
# starfish model

numx = 30
numy = 30
N = numx*numy
R = 0.5
nrec = 1000
n_j = nrec
wnum = int(n_j/2)+1

spacing = 1.2
dx = 0.4*spacing
dy = 0.4*spacing
Lx = spacing * 2 * R * numx
Ly = 0.5 * np.sqrt(3) * numy * spacing * 2 * R
xnum = round(Lx/dx)
ynum = round(Ly/dy)
dkx = 2*np.pi/Lx
dky = 2*np.pi/Ly
dt = 0.1
dw = 2*np.pi/(dt*n_j)

# frequency domain 
kxrange = np.zeros(xnum)
kyrange = np.zeros(ynum)
wrange = dw*np.arange(wnum)

for i in range(xnum):
    kxrange[i] = -np.pi/dx + dkx*i
for j in range(ynum):
    kyrange[j] = -np.pi/dy + dky*j
    
kx,ky = np.meshgrid(kxrange,kyrange,indexing='ij')


In [ ]:
# when dx was 0.5*spacing
#Gpoint = np.array([30,26],dtype=int)
#Mpoint = np.array([45,33],dtype=int)
#Kpoint = np.array([40,41],dtype=int)

# when dx was 0.4 spacing
Gpoint = np.array([37,32],dtype=int)
Mpoint = np.array([53,40],dtype=int)
Kpoint = np.array([48,47],dtype=int)

# when dx was 0.3 spacing
#Gpoint = np.array([50,43],dtype=int)
#Kpoint = np.array([60,58],dtype=int)
#Mpoint = np.array([65,51],dtype=int)

In [ ]:
base_dir = 'directory of data for alpha=0.79 and vsig=1'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_1 = np.fromfile(f,np.float64)
Cll_1 = (1/N)*Cll_1.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_1 = np.fromfile(f,np.float64)
Ctt_1 = (1/N)*Ctt_1.reshape((xnum,ynum,wnum))


base_dir = 'directory of data for alpha=2.63 and vsig=0.02'

file_name = 'Cll_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Cll_2 = np.fromfile(f,np.float64)
Cll_2 = (1/N)*Cll_2.reshape((xnum,ynum,wnum))

file_name = 'Ctt_starfish.dat'
full_path = os.path.join(base_dir,file_name)
with open(full_path,'rb') as f:
    Ctt_2 = np.fromfile(f,np.float64)
Ctt_2 = (1/N)*Ctt_2.reshape((xnum,ynum,wnum))


In [ ]:
ikx = Mpoint[0]
iky = Mpoint[1]

C1full = copy.copy(Cll_1) + copy.copy(Ctt_1)
C1full_q = copy.copy(C1full[ikx,iky,:])

C2full = copy.copy(Cll_2) + copy.copy(Ctt_2)
C2full_q = copy.copy(C2full[ikx,iky,:])


# coefficient of variation

In [ ]:
mu = 0 # mean
var = 0 # variance
Csum = 0
nonzeroCnum = 0

for i in range(wnum):
    if C1full_q[i]>0:
        Csum += C1full_q[i]
        nonzeroCnum += 1

for i in range(wnum):
    mu += wrange[i]*C1full_q[i]/Csum
    
for i in range(wnum):
    var += C1full_q[i]*((wrange[i]-mu)**2)/((nonzeroCnum-1)*Csum/nonzeroCnum)
    
CV1 = np.sqrt(var)/mu


In [ ]:
mu = 0 # mean
var = 0 # variance
Csum = 0
nonzeroCnum = 0

for i in range(wnum):
    if C2full_q[i]>0:
        Csum += C2full_q[i]
        nonzeroCnum += 1

for i in range(wnum):
    mu += wrange[i]*C2full_q[i]/Csum
    
for i in range(wnum):
    var += C2full_q[i]*((wrange[i]-mu)**2)/((nonzeroCnum-1)*Csum/nonzeroCnum)
    
CV2 = np.sqrt(var)/mu


# plot current correlation function at M point with CV

In [ ]:
%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(15,5))
ax1 = fig.add_subplot(1,2,1)
ax2 = fig.add_subplot(1,2,2)

ax1.plot(wrange,C1full_q,'bo')
ax1.text(0.7*wrange[-1],0.9*np.max(C1full_q),'CV = %s' % ("%.4f"%(CV1)),fontsize=20)
ax1.set_xlabel('$\omega$', fontsize=25)
ax1.set_ylabel('$C_{M}$', fontsize=25)
ax1.text(-6.5,220,'(a)',fontsize=25)
ax1.tick_params(labelsize=13)

ax2.plot(wrange,C2full_q,'bo')
ax2.text(0.7*wrange[-1],0.9*np.max(C2full_q),'CV = %s' % ("%.4f"%(CV2)),fontsize=20)
ax2.set_xlabel('$\omega$', fontsize=25)
ax2.set_ylabel('$C_{M}$', fontsize=25)
ax2.text(-6.5,0.147,'(b)',fontsize=25)
ax2.tick_params(labelsize=13)

plt.show()
#fig.savefig('CM_example_v1.pdf',dpi=fig.dpi,bbox_inches='tight')


# double axis plots of max C at M point and CV

In [ ]:
# the values of max C at M point and CV for different paramters have been obtained
# from performing the above analysis of current correlation functions and CVs for those parameters

vsig_f1 = np.array([0.02,0.04,0.06,0.08,0.1])
CM_f1 = np.array([0.13,0.5,1.1,2.1,3.2])
CV_f1 = np.array([0.5232,0.5235,0.5274,0.5252,0.5250])

f0_vsig1 = np.array([0.3,0.35,0.4,0.45,0.5])
kL = 1.9
alpha_vsig1 = f0_vsig1*5/kL
CM_vsig1 = np.array([110,113,120,123,130])
CV_vsig1 = np.array([0.7295,0.7166,0.6999,0.6870,0.6668])

%matplotlib inline

fig = plt.figure(facecolor=(1,1,1),figsize=(15,5))
ax1 = fig.add_subplot(1,2,1)
ax2 = fig.add_subplot(1,2,2)
ax3 = ax1.twinx()
ax4 = ax2.twinx()

# first figure
color = '#0072b2'
ax1.plot(alpha_vsig1,CM_vsig1,'o-',color=color,markersize=10)
ax1.plot([alpha_vsig1[0],alpha_vsig1[-1]],[0.5,0.5],'--',color=color)
ax1.set_xlabel(r'$\alpha$',fontsize=25)
ax1.set_ylabel('$C_{M}$',color=color,fontsize=25)
ax1.tick_params(axis='y',labelcolor=color,labelsize=15)
ax1.tick_params(axis='x',labelsize=15)
ax1.text(0.683,125,'(a)',fontsize=25)

color = '#d55e00'
ax3.plot(alpha_vsig1,CV_vsig1,'^-',color=color,markersize=10)
ax3.plot([alpha_vsig1[0],alpha_vsig1[-1]],[0.7,0.7],linestyle='dotted',color=color)
ax3.set_ylabel('$CV$',color=color,fontsize=25)
ax3.tick_params(axis='y',labelcolor=color,labelsize=15)

# second figure
color = '#0072b2'
ax2.plot(vsig_f1,CM_f1,'o-',color=color,markersize=10)
ax2.plot([vsig_f1[0],vsig_f1[-1]],[0.5,0.5],'--',color=color)
ax2.set_xlabel('$v_{\sigma}$',fontsize=25)
ax2.set_ylabel('$C_{M}$',color=color,fontsize=25)
ax2.tick_params(axis='y',labelcolor=color,labelsize=15)
ax2.tick_params(axis='x',labelsize=15)
ax2.text(0.0035,3.1,'(b)',fontsize=25)

#color = '#e69f00'
color = '#d55e00'
ax4.plot(vsig_f1,CV_f1,'^-',color=color,markersize=10)
ax4.plot([vsig_f1[0],vsig_f1[-1]],[0.7,0.7],linestyle='dotted',color=color)
ax4.set_ylabel('$CV$',color=color,fontsize=25)
ax4.tick_params(axis='y',labelcolor=color,labelsize=15)

fig.tight_layout()
plt.show()
#fig.savefig('CM_wave_v2.pdf',dpi=fig.dpi,bbox_inches='tight')
